In [1]:
import numpy as np
import pandas as pd

In [2]:
raw_path = './general_polymers.csv'

df = pd.read_csv(raw_path, low_memory=False)
df = df[df['smiles_2'].isna()]  # keep only homopolymers

# filter out bad values
# set thermal_conductivity to NaN if check_tc is False
tc_ok = df['check_tc'].astype(bool)
df.loc[~tc_ok, ['thermal_conductivity', 'thermal_diffusivity']] = pd.NA

In [3]:
# set Tg to NaN if 
# 1) outside min/max range,
# 2) rmse > 0.1
# 3) sigma(Tg) > 50 K
a_upper = df['tg_thermal_expansion_coef(upper_tg)']
a_below = df['tg_thermal_expansion_coef(below_tg)']
da = (a_upper - a_below).abs()

tg = df['tg']
rmse = df['tg_rmse']
Tmin = df['tg_min_temp']
Tmax = df['tg_max_temp']
dT = df['tg_interval_temp']

N = (Tmax - Tmin) / dT + 1.0
Sxx = (dT**2) * N * (N**2 - 1.0) / 12.0
Tbar = (Tmin + Tmax) / 2.0

Sxx = Sxx.where(Sxx > 0)
N = N.where(N > 1)

# Approx SEs
se_a = rmse / np.sqrt(Sxx)
se_b = rmse * np.sqrt((1.0 / N) + (Tbar**2) / Sxx)

sigma_da = np.sqrt(2.0) * se_a
sigma_db = np.sqrt(2.0) * se_b

# delta-method sigma(Tg) for Tg = Δb/Δa
sigma_tg = (1.0 / da) * np.sqrt((tg * sigma_da) ** 2 + (sigma_db) ** 2)

tg_ok = (
    (df['tg'] >= df['tg_min_temp'])
    & (df['tg'] <= df['tg_max_temp'])
    & (df['tg_rmse'] <= 0.1)
    & (sigma_tg <= 50)
)
num_valid = tg_ok.sum()
valid_fraction = num_valid / df['tg'].notna().sum()
print(f'Valid Tg values: {num_valid} ({valid_fraction:.2%})')
df.loc[~tg_ok, 'tg'] = pd.NA

Valid Tg values: 50525 (90.12%)


In [4]:
# estimate linear thermal expansion coefficient (CLTE) from rho-T curve at T=300K
T_ref = 300.0
above_tg = df['tg'] < T_ref

a = np.where(
    above_tg,
    df.get('tg_thermal_expansion_coef(upper_tg)'),
    df.get('tg_thermal_expansion_coef(below_tg)'),
)
b = np.where(
    above_tg,
    df.get('tg_thermal_expansion_intercept(upper_tg)'),
    df.get('tg_thermal_expansion_intercept(below_tg)'),
)
rho = a * T_ref + b
alpha = (-a / rho) / 3.0
df['CLTE'] = alpha
df.loc[~tg_ok, 'CLTE'] = pd.NA

In [5]:
# set self-diffusion to NaN if <= 0
log10_col = 'self-diffusion'
df.loc[df[log10_col] <= 0, log10_col] = pd.NA

In [6]:
prop_cols = [
    'thermal_conductivity', 'thermal_diffusivity', 'tg', 'CLTE',
    'density', 'Rg', 'self-diffusion', 'Cp', 'Cv',
    'qm_homo_monomer1', 'qm_lumo_monomer1', 'qm_dipole_monomer1', 'qm_polarizability_monomer1',
    'static_dielectric_const', 'refractive_index',
    'compressibility', 'isentropic_compressibility', 'bulk_modulus', 'isentropic_bulk_modulus',
]
# calculate mean property values for duplicate entries
df = df.groupby('smiles_list')[prop_cols].median().reset_index()

In [7]:
# remove extreme outliers using Tukey's outer fences (3*IQR)

# log transformations
data = df[prop_cols].copy() 
log10_col = ['self-diffusion']
data[log10_col] = np.log10(data[log10_col])
logm1_col = ['static_dielectric_const']
data[logm1_col] = np.log10(data[logm1_col] - 1)

median = data.median()
q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1
lower = q1 - 3 * iqr
upper = q3 + 3 * iqr

outlier = (data < lower) | (data > upper)
df[data.columns] = df[data.columns].mask(outlier)

data.mask(outlier, inplace=True)
scaled = (data - median) / iqr
scaled[log10_col + logm1_col]

,self-diffusion,static_dielectric_const
0,0.105738,-0.948391
1,0.169788,-1.197015
2,-0.952953,-0.850602
3,0.068263,-0.038510
4,0.056161,NaN
...,...,...
78373,0.302452,0.106566
78374,-0.365794,0.195342
78375,0.329717,0.413142
78376,0.410650,0.032273


In [8]:
df.to_csv('./cleaned.csv', index=False)
df[prop_cols]

,thermal_conductivity,thermal_diffusivity,tg,CLTE,density,Rg,self-diffusion,Cp,Cv,qm_homo_monomer1,qm_lumo_monomer1,qm_dipole_monomer1,qm_polarizability_monomer1,static_dielectric_const,refractive_index,compressibility,isentropic_compressibility,bulk_modulus,isentropic_bulk_modulus
0,0.222079,6.960000e-08,471.937847,0.000099,1.088499,20.194050,5.710000e-13,2919.030481,2842.288351,-8.867311,0.456783,3.758404,24.294569,1.116432,1.501486,6.290000e-10,6.125000e-10,1.593788e+09,1.636714e+09
1,0.153319,5.570000e-08,NaN,NaN,1.141800,20.621641,6.050000e-13,2411.071375,2376.835179,-7.773700,-0.347897,3.499876,36.385796,1.083171,1.630743,NaN,NaN,8.698592e+08,8.823242e+08
2,0.158620,6.230000e-08,NaN,NaN,1.109111,30.550344,2.195000e-13,2298.532146,2286.024325,-9.154042,0.396091,3.239450,16.567109,1.132903,1.457195,NaN,NaN,1.157983e+09,1.164241e+09
3,0.198649,7.400000e-08,NaN,NaN,1.118461,23.736163,5.520000e-13,2349.609307,2336.047817,-10.848437,0.667526,3.112952,17.620477,1.398788,1.351576,NaN,NaN,7.772877e+08,7.829523e+08
4,0.155442,5.080000e-08,NaN,NaN,1.020741,17.455724,5.460000e-13,2998.423249,2822.258238,NaN,-4.287795,9.279112,30.521911,NaN,1.443273,5.200000e-10,4.890000e-10,1.925220e+09,2.045656e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78373,0.261848,7.080000e-08,498.980526,0.000065,1.227024,23.514115,6.820000e-13,3013.597902,2974.292429,-8.301487,1.464831,3.356761,35.645315,1.485280,1.624381,2.210000e-10,2.190000e-10,4.516836e+09,4.576484e+09
78374,0.284106,7.600000e-08,375.951054,0.000116,1.155477,20.584666,3.730000e-13,3237.311098,3177.131289,-9.643815,2.450611,5.051234,26.428973,1.547217,1.473573,2.320000e-10,2.280000e-10,4.304804e+09,4.386281e+09
78375,0.271955,6.890000e-08,351.432762,0.000123,1.159006,21.429662,6.990000e-13,3407.965912,3285.944518,-9.288063,2.491451,7.148840,35.019926,1.734760,1.476892,2.070000e-10,2.000000e-10,4.833677e+09,5.013320e+09
78376,0.406919,1.140000e-07,NaN,NaN,1.278937,39.333458,7.520000e-13,2780.890603,2660.470503,NaN,NaN,NaN,NaN,1.438870,NaN,2.500000e-10,2.390000e-10,4.007562e+09,4.188949e+09
